# PQO — IEEE 37: PV, armazenamento e ilhamento

**Como o notebook está organizado**

| Seção | O que tem | Regra |
|---|---|---|
| 0. Setup | imports e caminhos | — |
| 1. Config | tudo que você edita (curvas, barras, limites, storage) | só dados |
| 2. Modelo | funções que montam o cenário no OpenDSS | só mandam comandos |
| 3. Motor | `run(cenario)` roda as 24 h e devolve DataFrames | sem variável global |
| 4. Métricas e plots | `metricas(res)`, `plot_*`, `comparar` | só leem `res` |
| 5. Partes I, II, III | dicionário de cenários + chamadas | curtas |

**Convenções de sinal**
- `P_sub_kW` > 0: importa da subestação; **< 0: fluxo reverso**.
- PV: positivo = gera. Storage: positivo = descarrega, negativo = carrega.
- Carregamento em pu: linha = I / NormAmps; trafo = corrente do enrolamento 1 / corrente nominal (pelo kVA).
- Perdas % = perdas / (energia das cargas + perdas).

**Premissas assumidas**
- `NormAmps` das linhas não vem do modelo, então vale o default do OpenDSS (400 A).
- `loadmult` multiplica a curva diária: "20% do pico" = 0,2 × curva.
- `pv_pu` = Pmpp total dos PVs em múltiplos da demanda base total, dividido igualmente entre as barras. A potência de pico efetiva sai abaixo do Pmpp por causa das curvas de temperatura e eficiência.
- Despacho do storage é fixo (curva em Config).

In [ ]:
# Rode uma vez se precisar instalar (depois reinicie o kernel):
# %pip install numpy pandas matplotlib dss-python

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dss import DSS            # OpenDSS (dss-python)

PASTA = Path(r"C:\Users\brun\Desktop\aff\bla")     # AJUSTAR
ARQUIVO_DSS = PASTA / "ieee37.dss"
os.chdir(PASTA)

DSSText = DSS.Text

## 1. Config

In [ ]:
# ---------- simulação ----------
T = 24            # passos (as curvas abaixo têm 24 pontos)
DT_H = 1.0        # horas por passo  ->  kWh = kW * DT_H

# ---------- limites ----------
V_MIN, V_MAX = 0.93, 1.05        # pu
CARREGAMENTO_MAX = 1.0           # pu
IGNORAR_LINHAS = ['jumper']      # fora das métricas de sobrecarga (minúsculo)

# ---------- curvas de carga (pu do pico) ----------
CARGAS_IND = ['s701a', 's701b', 's701c', 's728', 's737a', 's738a']
PERFIL_RESD = [0.45, 0.40, 0.37, 0.35, 0.35, 0.40, 0.55, 0.65, 0.60, 0.50, 0.45, 0.45,
               0.50, 0.48, 0.45, 0.45, 0.50, 0.65, 0.85, 0.95, 1.00, 0.95, 0.80, 0.60]
PERFIL_IND = [0.30, 0.28, 0.27, 0.27, 0.30, 0.35, 0.50, 0.70, 0.88, 0.95, 1.00, 1.00,
              0.90, 0.95, 1.00, 0.98, 0.92, 0.80, 0.65, 0.50, 0.42, 0.38, 0.34, 0.32]

# ---------- PV ----------
BARRAS_PV = ['728', '733', '737', '740', '718', '720', '722', '742']
PV_KV = 4.8
IRRAD = [0, 0, 0, 0, 0, 0, .1, .2, .3, .5, .8, .9, 1.0, 1.0, .99, .9, .7, .4, .1, 0, 0, 0, 0, 0]
TEMP = [25, 25, 25, 25, 25, 25, 25, 25, 35, 40, 45, 50, 60, 60, 55, 40, 35, 30, 25, 25, 25, 25, 25, 25]

# ---------- storage (despacho FIXO por enquanto: negativo carrega, positivo descarrega) ----------
DESPACHO_STO = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -0.3, -0.4, -0.45, -0.45, -0.45, -0.4,
                -0.15, 0.2, 0.4, 0.6, 0.6, 0.4, 0.2, 0]
STO_KV, STO_STORED, STO_RESERVE = 4.8, 20, 20
STORAGES = {                     # (nome, barra, kW, kWh)
    'central': [('StoCentral', '709', 2464, 10000)],
    'dist':    [('Sto1', '709', 1000, 4000),
                ('Sto2', '734', 1464, 6000)],
}

# ---------- Parte I / II ----------
PV_PU_VARREDURA = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 7.0, 8.0]
PV_PU_ALTO = 3.0                 # nível "elevado" da Parte II (ajustar depois da varredura)
LOADMULT_P2 = 0.2                # carga leve é onde sobra energia pro storage

# ---------- Parte III (pendente, só guardando os dados) ----------
CARGAS_ILHA = [('S736b', 42), ('S741c', 42), ('S732c', 42), ('S734c', 42), ('S735c', 85),
               ('S733a', 85), ('S740c', 85), ('S738a', 126), ('S737a', 140)]   # menor prioridade primeiro

## 2. Modelo (só comandos do OpenDSS)

In [ ]:
def ckt():
    return DSS.ActiveCircuit


def nomes(colecao):
    """AllNames do dss-python devolve ['NONE'] quando a coleção está vazia."""
    return [n for n in colecao.AllNames if n.upper() != 'NONE']


def compilar():
    DSSText.Command = 'Clear'
    DSSText.Command = f'Compile "{ARQUIVO_DSS.as_posix()}"'


def demanda_base_kw():
    ld = ckt().Loads
    total = 0.0
    for n in nomes(ld):
        ld.Name = n
        total += ld.kW
    return total


def set_loadshapes():
    resd = ' '.join(map(str, PERFIL_RESD))
    ind = ' '.join(map(str, PERFIL_IND))
    DSSText.Command = f'New LoadShape.resd_loadshape npts={len(PERFIL_RESD)} interval=1 mult=[{resd}]'
    DSSText.Command = f'New LoadShape.ind_loadshape npts={len(PERFIL_IND)} interval=1 mult=[{ind}]'
    industriais = {n.lower() for n in CARGAS_IND}
    for n in nomes(ckt().Loads):
        forma = 'ind_loadshape' if n.lower() in industriais else 'resd_loadshape'
        DSSText.Command = f'Edit Load.{n} daily={forma}'


def add_pv(pv_pu):
    pmpp = demanda_base_kw() * pv_pu / len(BARRAS_PV)
    irr = ' '.join(map(str, IRRAD))
    tmp = ' '.join(map(str, TEMP))
    DSSText.Command = 'New XYcurve.PvsT npts=4 xarray=[0 25 75 100] yarray=[1.2 1 .8 .6]'
    DSSText.Command = 'New XYcurve.Effic npts=4 xarray=[.1 .2 .4 1] yarray=[0.86 .9 .93 .97]'
    DSSText.Command = f'New LoadShape.Irrad npts={len(IRRAD)} interval=1 mult=[{irr}]'
    DSSText.Command = f'New Tshape.Temp npts={len(TEMP)} interval=1 temp=[{tmp}]'
    for i, barra in enumerate(BARRAS_PV, start=1):
        DSSText.Command = (
            f'New PVSystem.PV_{i} phases=3 bus1={barra}.1.2.3 kv={PV_KV} conn=delta '
            f'irrad=1.0 pmpp={pmpp:.3f} kva={pmpp:.3f} temperature=25 pf=1 '
            f'%cutin=0.05 %cutout=0.05 effcurve=Effic P-tcurve=PvsT Daily=Irrad Tdaily=Temp')
    return pmpp * len(BARRAS_PV)


def add_storage(chave):
    desp = ' '.join(map(str, DESPACHO_STO))
    DSSText.Command = 'New XYcurve.Eff npts=4 xarray=[.1 .2 .4 1] yarray=[0.86 .9 .93 .97]'
    DSSText.Command = f'New LoadShape.dispatch_shape npts={len(DESPACHO_STO)} interval=1 mult=[{desp}]'
    for nome, barra, kw, kwh in STORAGES[chave]:
        DSSText.Command = (
            f'New Storage.{nome} phases=3 bus1={barra}.1.2.3 kv={STO_KV} conn=delta '
            f'kWrated={kw} kWhrated={kwh} %stored={STO_STORED} %reserve={STO_RESERVE} '
            f'effcurve=Eff dispmode=follow daily=dispatch_shape pf=1')


def validar_cenario(sc):
    """O OpenDSS falha em silêncio; aqui conferimos se o que pedimos existe de fato."""
    c = ckt()
    esp_pv = len(BARRAS_PV) if sc.get('pv_pu', 0) > 0 else 0
    esp_st = len(STORAGES[sc['storage']]) if sc.get('storage') else 0
    got_pv, got_st = len(nomes(c.PVSystems)), len(nomes(c.Storages))
    if got_pv != esp_pv or got_st != esp_st:
        raise RuntimeError(f'Cenário não montou direito: PV {got_pv}/{esp_pv}, storage {got_st}/{esp_st}')


def montar_cenario(sc):
    """sc = dict(loadmult, pv_pu, storage ('central'|'dist'|None))"""
    if sc.get('ilha'):
        raise NotImplementedError('Parte III (ilhamento) ainda não implementada')
    compilar()
    set_loadshapes()
    DSSText.Command = f'set loadmult={sc["loadmult"]}'
    pmpp_total = add_pv(sc['pv_pu']) if sc.get('pv_pu', 0) > 0 else 0.0
    if sc.get('storage'):
        add_storage(sc['storage'])
    validar_cenario(sc)
    return pmpp_total

## 3. Motor

In [ ]:
def run(sc):
    """Roda um cenário (T passos) e devolve um dict de DataFrames."""
    pmpp_total = montar_cenario(sc)
    c = ckt()
    sol = c.Solution
    DSSText.Command = 'set controlmode=static'
    DSSText.Command = 'set mode=daily'
    DSSText.Command = f'set stepsize={DT_H}h'
    DSSText.Command = 'set number=1'

    n_lin = nomes(c.Lines)
    n_tr = [n for n in nomes(c.Transformers) if 'reg' not in n.lower()]
    n_ld, n_pv, n_st = nomes(c.Loads), nomes(c.PVSystems), nomes(c.Storages)

    # correntes nominais (estáticas)
    nom_lin = []
    for n in n_lin:
        c.Lines.Name = n
        nom_lin.append(c.Lines.NormAmps)
    nom_tr = []
    for n in n_tr:
        c.Transformers.Name = n
        c.Transformers.Wdg = 1
        nom_tr.append(c.Transformers.kVA / (np.sqrt(3) * c.Transformers.kV))

    def elem(nome):
        c.SetActiveElement(nome)
        return c.ActiveCktElement

    def i_max(el):                      # maior corrente de fase no terminal 1
        nf = el.NumPhases
        return max(el.CurrentsMagAng[0:2 * nf:2])

    def pq(el):                         # P, Q somados no terminal 1 (+ entra no elemento)
        nc, p = el.NumConductors, el.Powers
        return sum(p[0:2 * nc:2]), sum(p[1:2 * nc:2])

    V_rows = []
    I_lin, I_tr = np.zeros((T, len(n_lin))), np.zeros((T, len(n_tr)))
    P_ld, P_pv, P_st = np.zeros((T, len(n_ld))), np.zeros((T, len(n_pv))), np.zeros((T, len(n_st)))
    sub = np.zeros((T, 4))
    nao_conv = []

    for h in range(T):
        sol.Solve()
        if not sol.Converged:
            nao_conv.append(h)
        V_rows.append(np.array(c.AllBusVmagPu))
        tp, ls = c.TotalPower, c.Losses
        sub[h] = [-tp[0], -tp[1], ls[0] / 1000, ls[1] / 1000]
        for i, n in enumerate(n_lin):
            I_lin[h, i] = i_max(elem('Line.' + n))
        for i, n in enumerate(n_tr):
            I_tr[h, i] = i_max(elem('Transformer.' + n))
        for i, n in enumerate(n_ld):
            P_ld[h, i] = pq(elem('Load.' + n))[0]
        for i, n in enumerate(n_pv):
            P_pv[h, i] = -pq(elem('PVSystem.' + n))[0]
        for i, n in enumerate(n_st):
            P_st[h, i] = -pq(elem('Storage.' + n))[0]

    if nao_conv:
        print(f'ATENÇÃO: não convergiu nas horas {nao_conv}')

    idx = pd.RangeIndex(T, name='hora')
    nos = list(c.AllNodeNames)
    V = pd.DataFrame(np.vstack(V_rows), columns=nos, index=idx)
    V = V.loc[:, [n.split('.')[-1] in ('1', '2', '3') for n in nos]]

    S = pd.DataFrame(sub, columns=['P_sub_kW', 'Q_sub_kvar', 'perdas_kW', 'perdas_kvar'], index=idx)
    S['P_cargas_kW'] = P_ld.sum(axis=1)
    S['P_pv_kW'] = P_pv.sum(axis=1)
    S['P_sto_kW'] = P_st.sum(axis=1)
    S['balanco_kW'] = S.P_sub_kW + S.P_pv_kW + S.P_sto_kW - S.P_cargas_kW - S.perdas_kW   # deve ser ~0

    return dict(
        sc=sc, V=V,
        linhas=pd.DataFrame(I_lin / np.array(nom_lin), columns=n_lin, index=idx),
        trafos=pd.DataFrame(I_tr / np.array(nom_tr), columns=n_tr, index=idx),
        cargas=pd.DataFrame(P_ld, columns=n_ld, index=idx),
        pv=pd.DataFrame(P_pv, columns=n_pv, index=idx),
        sto=pd.DataFrame(P_st, columns=n_st, index=idx),
        sub=S, pmpp_total_kW=pmpp_total, nao_conv=nao_conv)

## 4. Métricas e plots

In [ ]:
def _linhas(res):
    L = res['linhas']
    return L.drop(columns=[c for c in IGNORAR_LINHAS if c in L.columns])


def metricas(res):
    V, L, Tr, S = res['V'], _linhas(res), res['trafos'], res['sub']
    sob, sub_ = V > V_MAX, V < V_MIN
    l_ex, t_ex = L > CARREGAMENTO_MAX, Tr > CARREGAMENTO_MAX
    rev = (-S.P_sub_kW).clip(lower=0)
    e_cargas = S.P_cargas_kW.sum() * DT_H
    e_perd = S.perdas_kW.sum() * DT_H
    return pd.Series({
        'Vmax_pu': V.max().max(),
        'Vmin_pu': V.min().min(),
        'sobretensao_no_h': int(sob.sum().sum()),
        'nos_sobretensao': int(sob.any().sum()),
        'subtensao_no_h': int(sub_.sum().sum()),
        'nos_subtensao': int(sub_.any().sum()),
        'linha_max_pu': L.max().max(),
        'sobrecarga_linha_h': int(l_ex.sum().sum()),
        'linhas_sobrecarregadas': int(l_ex.any().sum()),
        'trafo_max_pu': Tr.max().max(),
        'sobrecarga_trafo_h': int(t_ex.sum().sum()),
        'trafos_sobrecarregados': int(t_ex.any().sum()),
        'fluxo_reverso_kWh': rev.sum() * DT_H,
        'pico_reverso_kW': rev.max(),
        'horas_reverso': int((S.P_sub_kW < 0).sum() * DT_H),
        'E_cargas_kWh': e_cargas,
        'E_pv_kWh': S.P_pv_kW.sum() * DT_H,
        'perdas_kWh': e_perd,
        'perdas_pct': 100 * e_perd / (e_cargas + e_perd),
        'erro_balanco_max_kW': S.balanco_kW.abs().max(),
    })


def _ax_hora(ax, ylabel, titulo=''):
    ax.set_xlabel('hora'); ax.set_ylabel(ylabel); ax.set_title(titulo, fontsize=9)
    ax.set_xticks(range(0, T + 1, 3)); ax.grid(alpha=.3)


def plot_tensoes(res, titulo=''):
    V = res['V']
    fig, ax = plt.subplots(figsize=(8, 4))
    for k, cor in zip('123', ['tab:blue', 'tab:orange', 'tab:green']):
        cols = [c for c in V.columns if c.endswith('.' + k)]
        if not cols:
            continue
        vmx, vmn = V[cols].max(axis=1), V[cols].min(axis=1)
        ax.fill_between(V.index, vmn, vmx, color=cor, alpha=.25, label=f'Fase {"ABC"[int(k) - 1]}')
        ax.plot(V.index, vmx, color=cor, lw=.8); ax.plot(V.index, vmn, color=cor, lw=.8)
    ax.axhline(V_MAX, color='r', ls='--', lw=.8); ax.axhline(V_MIN, color='r', ls='--', lw=.8)
    ax.legend(fontsize=7, loc='upper left')
    _ax_hora(ax, 'pu', f'Tensão máx/mín por fase — {titulo}')
    plt.show()


def plot_carregamento(res, titulo=''):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    for ax, df, nome in [(axs[0], _linhas(res), 'Linhas'), (axs[1], res['trafos'], 'Transformadores')]:
        estoura = [c for c in df.columns if (df[c] > CARREGAMENTO_MAX).any()]
        for c in df.columns:
            if c not in estoura:
                ax.plot(df.index, df[c], color='lightgrey', lw=.8)
        for c in estoura:
            ax.plot(df.index, df[c], lw=1.4, label=c)
        ax.axhline(CARREGAMENTO_MAX, color='r', ls='--', lw=.8)
        if estoura:
            ax.legend(fontsize=6, ncol=2, loc='upper left')
        _ax_hora(ax, 'pu', f'{nome} — {titulo} (cinza = dentro do limite)')
    plt.show()


def plot_subestacao(res, titulo=''):
    S = res['sub']
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    axs[0].plot(S.index, S.P_sub_kW); axs[0].axhline(0, color='k', lw=.6)
    _ax_hora(axs[0], 'kW', f'P na subestação (<0 = reverso) — {titulo}')
    axs[1].plot(S.index, S.Q_sub_kvar); axs[1].axhline(0, color='k', lw=.6)
    _ax_hora(axs[1], 'kvar', 'Q na subestação')
    axs[2].plot(S.index, S.perdas_kW)
    _ax_hora(axs[2], 'kW', 'Perdas ativas')
    plt.show()


def plot_geracao(res, titulo=''):
    S = res['sub']
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    axs[0].plot(S.index, S.P_cargas_kW, label='Cargas')
    axs[0].plot(S.index, S.P_sub_kW, label='Subestação (+ importa)')
    if res['pv'].shape[1]:
        axs[0].plot(S.index, S.P_pv_kW, label='PV total')
    if res['sto'].shape[1]:
        axs[0].plot(S.index, S.P_sto_kW, label='Storage (+ descarrega)')
    axs[0].axhline(0, color='k', lw=.6); axs[0].legend(fontsize=7)
    _ax_hora(axs[0], 'kW', f'Balanço de potência — {titulo}')
    for c in res['pv']:
        axs[1].plot(res['pv'].index, res['pv'][c], lw=.9, label=c)
    for c in res['sto']:
        axs[1].plot(res['sto'].index, res['sto'][c], ls='--', label=c)
    if res['pv'].shape[1] or res['sto'].shape[1]:
        axs[1].legend(fontsize=6, ncol=2)
    _ax_hora(axs[1], 'kW', 'PV e storage individuais')
    plt.show()


def ver(res, titulo=''):
    """Os quatro plots de um cenário."""
    plot_tensoes(res, titulo); plot_carregamento(res, titulo)
    plot_subestacao(res, titulo); plot_geracao(res, titulo)


def comparar(RES):
    """Sobrepõe os cenários em seis painéis."""
    fig, axs = plt.subplots(2, 3, figsize=(15, 7))
    painel = [
        ('Tensão máxima (pu)', lambda r: r['V'].max(axis=1), V_MAX),
        ('Tensão mínima (pu)', lambda r: r['V'].min(axis=1), V_MIN),
        ('Carregamento máx. das linhas (pu)', lambda r: _linhas(r).max(axis=1), CARREGAMENTO_MAX),
        ('Carregamento máx. dos trafos (pu)', lambda r: r['trafos'].max(axis=1), CARREGAMENTO_MAX),
        ('P na subestação (kW, <0 = reverso)', lambda r: r['sub'].P_sub_kW, 0),
        ('Perdas ativas (kW)', lambda r: r['sub'].perdas_kW, None),
    ]
    for ax, (titulo, f, ref) in zip(axs.ravel(), painel):
        for nome, r in RES.items():
            ax.plot(f(r).index, f(r).values, label=nome)
        if ref is not None:
            ax.axhline(ref, color='r', ls='--', lw=.8)
        _ax_hora(ax, '', titulo)
    axs[0, 0].legend(fontsize=7)
    plt.tight_layout(); plt.show()


def varredura_pv(loadmult, niveis=None, storage=None):
    """Roda o cenário com PV crescente; devolve métricas indexadas por pv_pu."""
    linhas = []
    for pu in (PV_PU_VARREDURA if niveis is None else niveis):
        m = metricas(run(dict(loadmult=loadmult, pv_pu=float(pu), storage=storage)))
        m['pv_pu'] = pu
        linhas.append(m)
    return pd.DataFrame(linhas).set_index('pv_pu')


def limiares(df):
    """Primeiro nível de PV (× demanda) em que cada violação aparece."""
    regras = {
        'sobretensão': df.sobretensao_no_h > 0,
        'subtensão': df.subtensao_no_h > 0,
        'sobrecarga de linha': df.sobrecarga_linha_h > 0,
        'sobrecarga de trafo': df.sobrecarga_trafo_h > 0,
        'fluxo reverso': df.fluxo_reverso_kWh > 0,
    }
    return pd.Series({k: (df.index[v.values].min() if v.any() else np.nan) for k, v in regras.items()},
                     name='primeiro pv_pu com violação')


def plot_varredura(df, titulo=''):
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    axs[0].plot(df.index, df.Vmax_pu, marker='o'); axs[0].axhline(V_MAX, color='r', ls='--', lw=.8)
    axs[0].set_title(f'Tensão máxima — {titulo}', fontsize=9)
    axs[1].plot(df.index, df.linha_max_pu, marker='o', label='linhas')
    axs[1].plot(df.index, df.trafo_max_pu, marker='s', label='trafos')
    axs[1].axhline(CARREGAMENTO_MAX, color='r', ls='--', lw=.8); axs[1].legend(fontsize=7)
    axs[1].set_title('Carregamento máximo (pu)', fontsize=9)
    axs[2].plot(df.index, df.fluxo_reverso_kWh, marker='o')
    axs[2].set_title('Energia em fluxo reverso (kWh)', fontsize=9)
    for ax in axs:
        ax.set_xlabel('PV total (× demanda base)'); ax.grid(alpha=.3)
    plt.show()

## Checagem do modelo
Rode antes de tudo: confere nomes de barras, a demanda base e o `NormAmps` real das linhas.

In [ ]:
def checar_modelo():
    compilar()
    c = ckt()
    barras = [b.lower() for b in c.AllBusNames]
    print(f'Barras: {len(barras)} | cargas: {len(nomes(c.Loads))} | linhas: {len(nomes(c.Lines))} | trafos: {nomes(c.Transformers)}')
    print(f'Demanda base total: {demanda_base_kw():.1f} kW')
    na = set()
    for n in nomes(c.Lines):
        c.Lines.Name = n
        na.add(c.Lines.NormAmps)
    print('NormAmps das linhas (valores únicos):', sorted(na))
    print(f'PV: {len(BARRAS_PV)} barras de {len(barras)} = {100 * len(BARRAS_PV) / len(barras):.1f}%')
    todas = BARRAS_PV + [b for lst in STORAGES.values() for _, b, _, _ in lst]
    faltam = [b for b in todas if b.lower() not in barras]
    print('Barras inexistentes:', faltam if faltam else 'nenhuma')


checar_modelo()

## Parte I — PV distribuído (≥ 20% das barras) e cargas pico / 20%
Cenários 1 a 4. PV total = demanda base total, dividida entre as 8 barras.

In [ ]:
CENARIOS_P1 = {
    '1_semPV_pico': dict(loadmult=1.0, pv_pu=0.0, storage=None),
    '2_semPV_20%':  dict(loadmult=0.2, pv_pu=0.0, storage=None),
    '3_PV_20%':     dict(loadmult=0.2, pv_pu=1.0, storage=None),
    '4_PV_pico':    dict(loadmult=1.0, pv_pu=1.0, storage=None),
}
RES = {nome: run(sc) for nome, sc in CENARIOS_P1.items()}
tabela_p1 = pd.DataFrame({nome: metricas(r) for nome, r in RES.items()})
tabela_p1.round(3)

In [ ]:
comparar(RES)

In [ ]:
# detalhe de um cenário (troque a chave)
ver(RES['4_PV_pico'], '4_PV_pico')

### Cenário 4: aumentando o PV
Varredura de PV total (em múltiplos da demanda) até aparecerem violações. Rodo com carga de pico e com 20%.

In [ ]:
VARR_PICO = varredura_pv(1.0)
VARR_LEVE = varredura_pv(0.2)
print('Carga de pico:'); print(limiares(VARR_PICO))
print('\nCarga 20%:');   print(limiares(VARR_LEVE))
VARR_PICO.round(3)

In [ ]:
plot_varredura(VARR_PICO, 'carga de pico')
plot_varredura(VARR_LEVE, 'carga 20%')

### Discussão Parte I
*(preencher: o que aconteceu em cada cenário, onde aparece sobretensão, sobrecarga, fluxo reverso, efeito nas perdas)*

## Parte II — Armazenamento (central × distribuído)
Mesmo PV elevado (`PV_PU_ALTO`) e mesma carga (`LOADMULT_P2`); só muda o storage. Despacho fixo por enquanto.

In [ ]:
CENARIOS_P2 = {
    'A_PV_alto_sem_sto': dict(loadmult=LOADMULT_P2, pv_pu=PV_PU_ALTO, storage=None),
    'B_storage_central': dict(loadmult=LOADMULT_P2, pv_pu=PV_PU_ALTO, storage='central'),
    'C_storage_dist':    dict(loadmult=LOADMULT_P2, pv_pu=PV_PU_ALTO, storage='dist'),
}
RES2 = {nome: run(sc) for nome, sc in CENARIOS_P2.items()}
tabela_p2 = pd.DataFrame({nome: metricas(r) for nome, r in RES2.items()})
tabela_p2.round(3)

In [ ]:
comparar(RES2)

In [ ]:
ver(RES2['B_storage_central'], 'B_storage_central')
ver(RES2['C_storage_dist'], 'C_storage_dist')

### Discussão Parte II
*(preencher: central × distribuído; lembrar que o despacho é fixo e que o SoC ainda não é monitorado)*

## Parte III — Microrrede e ilhamento (pendente)
Decisões já tomadas:
- Referência de tensão na ilha: `Vsource` na barra do BESS (grid-forming), só no cenário de ilha; PV e carga são PQ e não seguram tensão.
- Tempo de sobrevivência = energia que o BESS entrega na ilha (carga − PV + perdas) contra kWh útil (descontando reserva e eficiência). Pode passar de 24 h, então o loop repete a curva diária até o SoC chegar na reserva.
- Corte de carga: `CARGAS_ILHA` (em Config) na ordem de prioridade, cortando as N primeiras.

A chave `ilha` do cenário já existe em `montar_cenario` e levanta `NotImplementedError` até isso ser feito.